# UCI Adult — baseline e comparação inicial

Nesta etapa ajustamos candidatos simples para estabelecer uma referência. O teste oficial permanece reservado para a avaliação final. `sex` e `race` são mantidos para auditoria e não entram inicialmente nas features.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'pt-BR' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from adult_data import load_adult
from baseline_models import write_baseline_outputs
frame = load_adult(ROOT / 'data' / 'raw' / 'adult_uci')
outputs = write_baseline_outputs(frame, ROOT / 'data' / 'processed' / 'baseline')
outputs

{'comparison': WindowsPath('d:/Projetos/ModelRisk_ResponsibleML/data/processed/baseline/baseline_model_comparison.csv'),
 'contract': WindowsPath('d:/Projetos/ModelRisk_ResponsibleML/data/processed/baseline/baseline_split_contract.json')}

## Contrato da divisão

O treino oficial foi dividido de forma estratificada em ajuste e calibração/validação. O `adult.test` não foi usado para seleção, ajuste de limiar ou comparação inicial.

In [2]:
import json
json.loads(Path(outputs['contract']).read_text(encoding='utf-8'))

{'official_train_rows': 32561,
 'fit_rows': 26048,
 'calibration_rows': 6513,
 'official_test_rows_reserved': 16281,
 'predictors': ['age',
  'workclass',
  'fnlwgt',
  'education',
  'education_num',
  'marital_status',
  'occupation',
  'relationship',
  'capital_gain',
  'capital_loss',
  'hours_per_week',
  'native_country'],
 'audit_columns_excluded_from_predictors': ['sex', 'race'],
 'random_state': 42,
 'selection_threshold': 0.5,
 'test_used': False}

## Comparação dos modelos

Pergunta orientadora: qual candidato estabelece o melhor ponto de partida considerando discriminação, qualidade probabilística e equilíbrio entre métricas?

As métricas abaixo são calculadas na calibração/validação, usando o limiar provisório de 0,5 apenas para as métricas de classificação. ROC-AUC e PR-AUC avaliam a ordenação dos casos; Brier score avalia a qualidade das probabilidades. A seleção definitiva ainda não foi feita.

In [3]:
import pandas as pd
comparison = pd.read_csv(outputs['comparison'])
comparison

,model,roc_auc,pr_auc,accuracy,balanced_accuracy,precision,recall,f1,brier
0,hist_gradient_boosting,0.928213,0.833058,0.874098,0.800148,0.784627,0.657526,0.715475,0.087381
1,random_forest,0.919201,0.807758,0.868571,0.789104,0.777691,0.635842,0.699649,0.094590
2,logistic_regression,0.906815,0.772540,0.855520,0.772234,0.742835,0.611607,0.670864,0.101257
3,dummy_prior,0.500000,0.240749,0.759251,0.500000,0.000000,0.000000,0.000000,0.182789


### Insights:

O `DummyClassifier` estabelece o piso de comparação. Um modelo útil precisa superar esse piso em discriminação e qualidade das probabilidades. O `HistGradientBoosting` alcançou ROC-AUC de 0,928213 e PR-AUC de 0,833058 na calibração/validação, um resultado forte para esta etapa e claramente superior à regressão logística (ROC-AUC de 0,906815) e ao Random Forest (0,919201).

Esse 0,928213 é motivo para entusiasmo, mas deve ser interpretado como evidência de boa capacidade de ordenação neste split de calibração, não como desempenho final garantido. O modelo ainda precisa ser avaliado quanto à calibração, estabilidade, comportamento por sexo e raça e desempenho no teste oficial reservado.

Nenhuma métrica isolada define o modelo escolhido, e esta comparação ainda não permite concluir qual modelo é aceitável por grupo. A auditoria de fairness será feita depois, nos candidatos que avançarem, sem consumir o teste oficial antecipadamente.

## Próximas perguntas

1. Como os candidatos se comportam por sexo, raça e interseção?
2. As probabilidades precisam de calibração?
3. Qual modelo oferece o melhor compromisso entre desempenho, estabilidade e interpretabilidade?
4. Qual limiar será justificável sem usar o teste?

Essas perguntas serão tratadas nas etapas de calibração e auditoria.